In [3]:
import sys, torch
sys.path.insert(0, ".")

from DNABERT2_modules import load_dnabert2
import transformers

print(f"transformers version: {transformers.__version__}")
tokenizer = transformers.AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", 
                                                       use_fast=False, trust_remote_code=True)


# Load pretrained weights into the local BertModel class.
# config_overrides lets you modify any BertConfig field before instantiation.
model, tokenizer = load_dnabert2(
    # config_overrides=None,   # e.g. {"num_hidden_layers": 6, "hidden_dropout_prob": 0.1}
    add_pooling_layer=False,
    config_overrides={"pad_token_id": tokenizer.pad_token_id}
)

device = next(model.parameters()).device
print(f"device     : {device}")
print(f"hidden_size: {model.config.hidden_size}")
print(f"num_layers : {model.config.num_hidden_layers}")

transformers version: 4.42.4
device     : cuda:0
hidden_size: 768
num_layers : 12


In [4]:
dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
inputs = tokenizer(dna, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

In [5]:
with torch.no_grad():
    hidden_states, _ = model(**inputs)  # (1, seq_len, 768)

# mean-pooled embedding (mask-weighted)
mask = inputs["attention_mask"].unsqueeze(-1).float()
embedding_mean = (hidden_states * mask).sum(1) / mask.sum(1).clamp(min=1)
print(embedding_mean.shape)  # expect (1, 768)

# token-level hidden states
print(hidden_states.shape)  # expect (1, seq_len, 768)

torch.Size([1, 768])
torch.Size([1, 17, 768])


In [6]:
import pathlib
_cache_dir = pathlib.Path("./output_cache")
_cache_dir.mkdir(exist_ok=True)
torch.save(hidden_states.cpu(), _cache_dir / "modern_hidden_states.pt")
torch.save(embedding_mean.cpu(), _cache_dir / "modern_embedding_mean.pt")
print(f"modern outputs cached → {_cache_dir.resolve()}")

modern outputs cached → /home/andrew.dickson/svar/output_cache


In [1]:
# RUN WITH SVAR_LEGACY_39 CODE
import torch
import transformers
from transformers import AutoModel

print(f"transformers version: {transformers.__version__}")
device = torch.device('cuda')
tokenizer = transformers.AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M",
                                                       use_fast=False, trust_remote_code=True)
dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
inputs = tokenizer(dna, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Only works for old versions of pytorch and transformers
ref_model = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
ref_model = ref_model.to(device).eval()

# AutoModel uses the broken Triton path; patch it the same way our loader does.
from DNABERT2_modules.loader import _disable_triton_attention
_disable_triton_attention(ref_model)

with torch.no_grad():
    ref_hidden, _ = ref_model(**inputs)

mask = inputs["attention_mask"].unsqueeze(-1).float()
ref_embedding_mean = (ref_hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
print(f"ref_hidden shape:         {ref_hidden.shape}")
print(f"ref_embedding_mean shape: {ref_embedding_mean.shape}")

transformers version: 4.29.2


/home/andrew.dickson/.conda/envs/svar_legacy_39/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at zhihan1996/DNABERT-2-117M were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect t

ref_hidden shape:         torch.Size([1, 17, 768])
ref_embedding_mean shape: torch.Size([1, 768])


In [2]:
import pathlib
_cache_dir = pathlib.Path("./output_cache")
_cache_dir.mkdir(exist_ok=True)
torch.save(ref_hidden.cpu(), _cache_dir / "reference_hidden_states.pt")
torch.save(ref_embedding_mean.cpu(), _cache_dir / "reference_embedding_mean.pt")
print(f"reference outputs cached → {_cache_dir.resolve()}")

reference outputs cached → /home/andrew.dickson/svar/output_cache


In [10]:
import pathlib
_cache_dir = pathlib.Path("./output_cache")
_cache_dir.mkdir(exist_ok=True)
ref_hidden = torch.load(_cache_dir / "reference_hidden_states.pt")
ref_embedding_mean = torch.load(_cache_dir / "reference_embedding_mean.pt")
modern_hidden = torch.load(_cache_dir / "modern_hidden_states.pt")
modern_embedding_mean = torch.load(_cache_dir / "modern_embedding_mean.pt")
print(f"reference outputs cached → {_cache_dir.resolve()}")

print("Max diff:", (modern_hidden - ref_hidden).abs().max())


reference outputs cached → /home/andrew.dickson/svar/output_cache
Max diff: tensor(0.)
